# Erasus — Audio Model Unlearning (Whisper-small, Tesla T4)

This notebook demonstrates **coreset-based machine unlearning** on an audio
foundation model — OpenAI's **Whisper-small** (244M parameters).

We fine-tune the Whisper encoder as an **audio classifier** on synthetic mel
spectrograms (10 sound classes), then use [Erasus](https://github.com/OnePunchMonk/erasus)
to surgically **unlearn specific sound classes** via gradient ascent while
varying:

- **Coreset selector** — 5 methods (`gradient_norm`, `el2n`, `herding`, `kcenter`, `random`)
- **Coreset fraction** — 6 values from 1% to 100% of the forget set

The result is a comprehensive ablation of how selector quality and coreset size
affect the forget/retain accuracy trade-off on a real audio backbone.

**Runtime:** ~20-30 min on a free Colab T4 GPU (15 GB VRAM).

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git matplotlib
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import time
import json
import warnings
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import WhisperModel

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------------------------------------------------------ #
# Load Whisper-small encoder
# ------------------------------------------------------------------ #

whisper = WhisperModel.from_pretrained("openai/whisper-small")
encoder = whisper.encoder.to(device)
del whisper.decoder  # free decoder memory
print(f"Whisper-small encoder loaded — {sum(p.numel() for p in encoder.parameters()) / 1e6:.1f}M params")

# ------------------------------------------------------------------ #
# WhisperClassifier: encoder + linear head
# ------------------------------------------------------------------ #

class WhisperClassifier(nn.Module):
    """Whisper encoder with a classification head on pooled output."""

    def __init__(self, encoder, hidden_dim=768, n_classes=10):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, labels=None, **kwargs):
        # x: [B, 80, 3000] mel spectrogram
        enc_out = self.encoder(x).last_hidden_state  # [B, T, 768]
        pooled = enc_out.mean(dim=1)                 # [B, 768]
        logits = self.head(pooled)                   # [B, n_classes]
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            return type("Out", (), {"logits": logits, "loss": loss})()
        return logits

# Freeze encoder, only train head initially
for p in encoder.parameters():
    p.requires_grad = False

model = WhisperClassifier(encoder, hidden_dim=768, n_classes=10).to(device)
print(f"WhisperClassifier created — trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e3:.1f}K")

# ------------------------------------------------------------------ #
# Synthetic mel spectrogram data
# ------------------------------------------------------------------ #

torch.manual_seed(42)

def make_mel_data(n_samples, classes, mel_bins=80, mel_frames=3000):
    """Generate synthetic mel spectrograms with class-specific frequency bias."""
    samples_per_class = n_samples // len(classes)
    all_mels, all_labels = [], []
    for c in classes:
        mels = torch.randn(samples_per_class, mel_bins, mel_frames) * 0.3
        # Class-specific bias: boost mel bins c*8:(c+1)*8 by +2.0
        lo, hi = c * 8, (c + 1) * 8
        if hi <= mel_bins:
            mels[:, lo:hi, :] += 2.0
        all_mels.append(mels)
        all_labels.append(torch.full((samples_per_class,), c, dtype=torch.long))
    return torch.cat(all_mels), torch.cat(all_labels)

# Forget: 200 samples from classes 0, 1
forget_mels, forget_labels = make_mel_data(200, classes=[0, 1])
# Retain: 600 samples from classes 2-9
retain_mels, retain_labels = make_mel_data(600, classes=list(range(2, 10)))

print(f"Forget set: {forget_mels.shape[0]} samples (classes 0, 1)")
print(f"Retain set: {retain_mels.shape[0]} samples (classes 2-9)")
print(f"Mel shape: {list(forget_mels.shape[1:])}")

forget_set = TensorDataset(forget_mels, forget_labels)
retain_set = TensorDataset(retain_mels, retain_labels)
train_mels = torch.cat([forget_mels, retain_mels])
train_labels = torch.cat([forget_labels, retain_labels])
train_set = TensorDataset(train_mels, train_labels)

forget_loader = DataLoader(forget_set, batch_size=8, shuffle=True)
retain_loader = DataLoader(retain_set, batch_size=8, shuffle=True)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

# ------------------------------------------------------------------ #
# Train classifier (head only) for 5 epochs
# ------------------------------------------------------------------ #

optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-4)

print("\nTraining classifier head for 5 epochs ...")
for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for mels, labels in train_loader:
        mels, labels = mels.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(mels, labels=labels)
        out.loss.backward()
        optimizer.step()
        running_loss += out.loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"  Epoch {epoch} | loss {avg_loss:.4f}")

# Unfreeze encoder for unlearning (so gradient-based selectors work)
for p in model.encoder.parameters():
    p.requires_grad = True

# ------------------------------------------------------------------ #
# Evaluate base model
# ------------------------------------------------------------------ #

def compute_accuracy(mdl, loader, dev):
    """Return accuracy (0-1) of model on loader."""
    mdl.eval()
    correct = total = 0
    with torch.no_grad():
        for mels, labels in loader:
            mels, labels = mels.to(dev), labels.to(dev)
            logits = mdl(mels)
            if hasattr(logits, "logits"):
                logits = logits.logits
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

base_forget_acc = compute_accuracy(model, forget_loader, device)
base_retain_acc = compute_accuracy(model, retain_loader, device)
print(f"\nBase model  —  Forget acc: {base_forget_acc:.4f} | Retain acc: {base_retain_acc:.4f}")

# Snapshot trained state
trained_state = deepcopy(model.state_dict())
print("Saved trained state for ablation.")

In [ ]:
import erasus.strategies  # noqa: F401
import erasus.selectors   # noqa: F401
from erasus.unlearners import ErasusUnlearner

# ------------------------------------------------------------------ #
# Coreset ablation grid
# ------------------------------------------------------------------ #

FRACTIONS = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
SELECTORS = ["gradient_norm", "el2n", "herding", "kcenter", "random"]

results = {}  # {selector: {fraction: {forget_acc, retain_acc, time_s}}}

total_runs = len(SELECTORS) * len(FRACTIONS)
run_idx = 0

for selector_name in SELECTORS:
    results[selector_name] = {}
    for frac in FRACTIONS:
        run_idx += 1
        tag = f"[{run_idx}/{total_runs}]"

        # Fresh copy of the trained model
        m = WhisperClassifier(encoder, hidden_dim=768, n_classes=10).to(device)
        m.load_state_dict(deepcopy(trained_state))
        # Ensure all params require grad for selectors
        for p in m.parameters():
            p.requires_grad = True

        try:
            unlearner = ErasusUnlearner(
                model=m,
                strategy="gradient_ascent",
                selector=selector_name,
            )
            t0 = time.time()
            unlearner.fit(
                forget_data=forget_loader,
                retain_data=retain_loader,
                prune_ratio=frac,
                epochs=5,
            )
            elapsed = time.time() - t0

            f_acc = compute_accuracy(unlearner.model, forget_loader, device)
            r_acc = compute_accuracy(unlearner.model, retain_loader, device)
        except Exception as exc:
            print(f"{tag}  {selector_name:>15s}  frac={frac:.2f}  —  ERROR: {exc}")
            f_acc = float("nan")
            r_acc = float("nan")
            elapsed = 0.0

        results[selector_name][frac] = {
            "forget_acc": f_acc,
            "retain_acc": r_acc,
            "time_s": round(elapsed, 2),
        }
        print(
            f"{tag}  {selector_name:>15s}  frac={frac:.2f}  |  "
            f"forget={f_acc:.4f}  retain={r_acc:.4f}  ({elapsed:.1f}s)"
        )

# ------------------------------------------------------------------ #
# Summary table
# ------------------------------------------------------------------ #

print("\n" + "=" * 80)
print(f"{'Selector':>15s}  {'Frac':>5s}  {'Forget':>8s}  {'Retain':>8s}  {'Time(s)':>8s}")
print("-" * 80)
for sel in SELECTORS:
    for frac in FRACTIONS:
        r = results[sel][frac]
        print(
            f"{sel:>15s}  {frac:>5.2f}  "
            f"{r['forget_acc']:>8.4f}  {r['retain_acc']:>8.4f}  {r['time_s']:>8.2f}"
        )
    print()
print("=" * 80)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------------ #
# Prepare arrays
# ------------------------------------------------------------------ #

fracs = np.array(FRACTIONS)

forget_curves = {}
retain_curves = {}
trade_curves  = {}

for sel in SELECTORS:
    fa = np.array([results[sel][f]["forget_acc"] for f in FRACTIONS])
    ra = np.array([results[sel][f]["retain_acc"] for f in FRACTIONS])
    forget_curves[sel] = fa
    retain_curves[sel] = ra
    # Tradeoff: higher is better (maximize forgetting while keeping retain)
    trade_curves[sel] = (1.0 - fa) * ra

# ------------------------------------------------------------------ #
# 3-panel figure
# ------------------------------------------------------------------ #

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for sel in SELECTORS:
    axes[0].plot(fracs, forget_curves[sel], marker="o", label=sel)
    axes[1].plot(fracs, retain_curves[sel], marker="o", label=sel)
    axes[2].plot(fracs, trade_curves[sel], marker="o", label=sel)

axes[0].set_title("Forget Accuracy vs Coreset Fraction")
axes[0].set_xlabel("Coreset Fraction")
axes[0].set_ylabel("Forget Accuracy")
axes[0].axhline(base_forget_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Retain Accuracy vs Coreset Fraction")
axes[1].set_xlabel("Coreset Fraction")
axes[1].set_ylabel("Retain Accuracy")
axes[1].axhline(base_retain_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Tradeoff Score  (1 - forget) * retain")
axes[2].set_xlabel("Coreset Fraction")
axes[2].set_ylabel("Tradeoff Score")
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

fig.suptitle("Whisper-small Audio Unlearning — Coreset Ablation (Tesla T4)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("/content/whisper_coreset_ablation_3panel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved /content/whisper_coreset_ablation_3panel.png")

# ------------------------------------------------------------------ #
# Summary table (best fraction per selector)
# ------------------------------------------------------------------ #

print("\nBest tradeoff score per selector:")
print(f"{'Selector':>15s}  {'Best Frac':>9s}  {'Forget':>8s}  {'Retain':>8s}  {'Tradeoff':>9s}")
print("-" * 60)
for sel in SELECTORS:
    best_idx = int(np.nanargmax(trade_curves[sel]))
    best_frac = FRACTIONS[best_idx]
    r = results[sel][best_frac]
    ts = trade_curves[sel][best_idx]
    print(
        f"{sel:>15s}  {best_frac:>9.2f}  "
        f"{r['forget_acc']:>8.4f}  {r['retain_acc']:>8.4f}  {ts:>9.4f}"
    )

In [ ]:
import numpy as np

# ------------------------------------------------------------------ #
# Save results as JSON
# ------------------------------------------------------------------ #

output = {
    "experiment": "whisper_small_audio_coreset_ablation",
    "model": "WhisperClassifier (whisper-small encoder + linear head)",
    "model_params_M": 244,
    "dataset": "synthetic_mel_spectrograms",
    "mel_shape": [80, 3000],
    "forget_classes": [0, 1],
    "forget_samples": 200,
    "retain_samples": 600,
    "train_epochs": 5,
    "unlearn_epochs": 5,
    "strategy": "gradient_ascent",
    "base_forget_acc": base_forget_acc,
    "base_retain_acc": base_retain_acc,
    "fractions": FRACTIONS,
    "selectors": SELECTORS,
    "results": {},
}

for sel in SELECTORS:
    output["results"][sel] = {}
    for frac in FRACTIONS:
        r = results[sel][frac]
        output["results"][sel][str(frac)] = {
            "forget_acc": round(r["forget_acc"], 6) if not np.isnan(r["forget_acc"]) else None,
            "retain_acc": round(r["retain_acc"], 6) if not np.isnan(r["retain_acc"]) else None,
            "time_s": r["time_s"],
        }

json_path = "/content/whisper_coreset_ablation_results.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {json_path}")

# Download files (Colab-specific)
try:
    from google.colab import files
    files.download(json_path)
    files.download("/content/whisper_coreset_ablation_3panel.png")
except ImportError:
    print("Not running on Colab — files saved locally under /content/")